# 🔍 02 — Embeddings & Retrieval Quality Analysis

This notebook validates the output of the embedding and retrieval pipeline.
Use it to:
- Visualise the FAISS index with UMAP (2D projection)
- Test retrieval quality with sample queries
- Compare FAISS-only vs FAISS+BM25 hybrid vs Cohere rerank results
- Inspect BM25 lexical search on financial keywords

**Run this after `python -m src.embeddings.embeddings`**

In [ ]:
import os
import pickle
import sys
from pathlib import Path

import faiss
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from dotenv import load_dotenv

sys.path.insert(0, str(Path('..').resolve()))
load_dotenv('../.env')

from src.retrieval.retriever import (
    load_index, retrieve, embed_query,
    faiss_search, hybrid_search, rerank_all,
)
from src.retrieval.bm25_index import bm25_search

INDEX_DIR = Path('../data/index')
COLORS = {'tesla': '#E31937', 'apple': '#555555', 'microsoft': '#00A4EF'}

## 1. Load index

In [ ]:
# load_index() now returns (faiss_index, metadata, bm25_index)
index, metadata, bm25 = load_index()

print(f"FAISS vectors : {index.ntotal}")
print(f"Chunks        : {len(metadata)}")
print(f"Embedding dim : {index.d}")

df = pd.DataFrame(metadata)
print("\nChunks per company:")
print(df.groupby('company').size())
print("\nChunks per year:")
print(df.groupby('year').size())

## 2. UMAP visualisation

In [ ]:
try:
    import umap
except ImportError:
    print("Install umap-learn: pip install umap-learn")
    raise

# Extract all vectors from FAISS
vectors = faiss.rev_swig_ptr(index.get_xb(), index.ntotal * index.d)
vectors = np.array(vectors).reshape(index.ntotal, index.d)

print(f"Running UMAP on {len(vectors)} vectors...")
reducer  = umap.UMAP(n_components=2, random_state=42, n_neighbors=15, min_dist=0.1)
embedded = reducer.fit_transform(vectors)
print("Done.")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# — By company
ax = axes[0]
for company, color in COLORS.items():
    mask = df['company'] == company
    ax.scatter(embedded[mask, 0], embedded[mask, 1],
               c=color, label=company.capitalize(), alpha=0.5, s=8)
ax.set_title('UMAP — by company', fontsize=13)
ax.legend(markerscale=3)
ax.set_xlabel('UMAP-1'); ax.set_ylabel('UMAP-2')

# — By year
ax = axes[1]
years  = sorted(df['year'].unique())
cmap   = plt.cm.get_cmap('viridis', len(years))
y_min, y_max = min(years), max(years)
for i, year in enumerate(years):
    mask = df['year'] == year
    ax.scatter(embedded[mask, 0], embedded[mask, 1],
               c=[cmap(i / max(len(years)-1, 1))], label=str(year), alpha=0.5, s=8)
ax.set_title('UMAP — by year', fontsize=13)
ax.legend(markerscale=3, title='Year')
ax.set_xlabel('UMAP-1'); ax.set_ylabel('UMAP-2')

plt.tight_layout()
plt.savefig('../data/umap_embeddings.png', dpi=150, bbox_inches='tight')
plt.show()

## 3. Retrieval quality — sample queries

In [ ]:
TEST_QUERIES = [
    ("What are Tesla's main risk factors?",             None,          None),
    ("Apple Services segment revenue growth",           "apple",       None),
    ("Azure cloud revenue Microsoft",                   "microsoft",   2024),
    ("Compare R&D spending across all three companies", None,          None),
    ("How has Tesla AI strategy evolved over the years?",None,         None),
    ("iPhone revenue 2024",                             None,          None),  # BM25 test
    ("EBITDA margin",                                   None,          None),  # BM25 test
]

rows = []
for query, company_filter, year_filter in TEST_QUERIES:
    chunks = retrieve(query, index, metadata, bm25,
                      company_filter=company_filter,
                      year_filter=year_filter)
    for rank, chunk in enumerate(chunks, 1):
        rows.append({
            'query':   query[:50],
            'rank':    rank,
            'company': chunk['company'],
            'year':    chunk.get('year'),
            'pages':   f"{chunk['start_page']}–{chunk['end_page']}",
            'rerank':  round(chunk.get('rerank_score', chunk.get('rrf_score', 0)), 4),
            'text':    chunk['text'][:100] + '…',
        })

results_df = pd.DataFrame(rows)
results_df

## 4. FAISS-only vs BM25-only vs Hybrid (RRF) comparison

In [ ]:
def compare_retrievers(query: str, top_k: int = 5):
    """Compare FAISS, BM25, and hybrid results side by side."""
    faiss_results  = faiss_search(query, index, metadata, top_k=top_k)
    bm25_results   = bm25_search(query, bm25, metadata, top_k=top_k)
    hybrid_results = hybrid_search(query, index, metadata, bm25, top_k=top_k)

    def to_df(results, score_key):
        return pd.DataFrame([{
            'company': r['company'],
            'year':    r.get('year'),
            'pages':   f"{r['start_page']}–{r['end_page']}",
            'score':   round(r.get(score_key, 0), 4),
            'text':    r['text'][:80] + '…',
        } for r in results])

    print(f"\n{'='*60}")
    print(f"Query: {query}")
    print(f"{'='*60}")
    print("\n--- FAISS (semantic) ---")
    print(to_df(faiss_results, 'faiss_score').to_string(index=False))
    print("\n--- BM25 (lexical) ---")
    print(to_df(bm25_results, 'bm25_score').to_string(index=False))
    print("\n--- Hybrid RRF ---")
    print(to_df(hybrid_results, 'rrf_score').to_string(index=False))

# Queries where BM25 adds value (exact numbers/names)
compare_retrievers("iPhone revenue 2024")
compare_retrievers("Azure grew 29%")
compare_retrievers("EBITDA")

## 5. Company balance check (comparison queries)

In [ ]:
comparison_queries = [
    "Compare cybersecurity risk management across all three companies",
    "Compare R&D spending across Tesla, Apple, and Microsoft",
    "Compare cloud strategies of Apple and Microsoft",
]

fig, axes = plt.subplots(1, len(comparison_queries), figsize=(14, 4))

for ax, query in zip(axes, comparison_queries):
    chunks = retrieve(query, index, metadata, bm25)
    counts = pd.Series([c['company'] for c in chunks]).value_counts()
    colors = [COLORS.get(c, '#ccc') for c in counts.index]
    counts.plot(kind='bar', ax=ax, color=colors, edgecolor='white')
    ax.set_title(query[:40] + '…', fontsize=9)
    ax.set_xlabel('')
    ax.set_ylabel('chunks')
    ax.tick_params(axis='x', rotation=30)

plt.suptitle('Company balance in comparison query results', y=1.02, fontsize=12)
plt.tight_layout()
plt.savefig('../data/company_balance.png', dpi=150, bbox_inches='tight')
plt.show()

## 6. Rerank quality — score distribution

In [ ]:
import os
if not os.getenv('COHERE_API_KEY'):
    print("COHERE_API_KEY not set — skipping rerank comparison.")
else:
    query      = "What are the main cybersecurity risks?"
    candidates = hybrid_search(query, index, metadata, bm25, top_k=20)
    reranked   = rerank_all(query, candidates)

    before_scores = [c.get('rrf_score', 0) for c in candidates[:10]]
    after_scores  = [c.get('rerank_score', 0) for c in reranked[:10]]

    x = range(1, 11)
    plt.figure(figsize=(10, 4))
    plt.plot(x, before_scores, 'o--', label='RRF score (pre-rerank)', color='#8b949e')
    plt.plot(x, after_scores,  's-',  label='Cohere rerank score',    color='#e6b450')
    plt.xlabel('Rank')
    plt.ylabel('Score')
    plt.title(f'RRF vs Cohere rerank — top 10\n"{query}"')
    plt.legend()
    plt.tight_layout()
    plt.savefig('../data/rerank_comparison.png', dpi=150, bbox_inches='tight')
    plt.show()

    print("\nTop 5 after rerank:")
    for i, c in enumerate(reranked[:5], 1):
        print(f"  [{i}] {c['company']:10s} year={c.get('year')} "
              f"pages {c['start_page']}–{c['end_page']} "
              f"score={c['rerank_score']:.4f}")